# NumPy — Class 2 Numpy Advance
### Class date: 31 October 2026

This notebook continues the previous NumPy class. It follows the supplied class notebook and recording, organizes the material into a step-by-step learning resource, and includes dry runs, practical examples, exercises, and solutions.

**Run order:** Run the code cells from top to bottom. Random values and timing measurements will vary between runs and machines.

**Source handling:** The original notebook and recording are the primary sources. Where a spoken explanation was technically inaccurate or simplified, a clearly labelled clarification is included rather than silently changing what was taught.

## 1. `np.empty()` — allocate an array without initializing its values

The previous class introduced `np.zeros()`, `np.ones()`, and other array-creation functions. This class begins with `np.empty()`.

`np.empty(shape, dtype=...)` creates an array of the requested shape and dtype, but it does **not** fill the array with zeros or another known value. The elements contain whatever values happen to be in the allocated memory.

The word *empty* can be misleading: the array has a valid shape and contains values, but those values are uninitialized and must not be relied on.

In [ ]:
import numpy as np
import time

empty_int = np.empty((2, 2), dtype=np.int8)
print(empty_int)
print("shape:", empty_int.shape)
print("dtype:", empty_int.dtype)

The output may look like zeros, small integers, or other values. It can change between runs. This does not mean `np.empty()` is broken; it means the memory was not initialized.

**Rule:** Use `np.empty()` only when your program will overwrite every element before reading it. If you need known starting values, use `np.zeros()`, `np.ones()`, or `np.full()`.

### Why can `np.empty()` be faster?

- `np.zeros()` allocates memory and initializes every element to zero.
- `np.ones()` allocates memory and initializes every element to one.
- `np.empty()` allocates memory but skips the initialization step.

Skipping initialization can save time when you are about to fill the whole array yourself. For small arrays, the difference may be tiny or dominated by measurement noise.

In [ ]:
start_empty = time.perf_counter()
empty_array = np.empty((500, 500))
empty_seconds = time.perf_counter() - start_empty

start_ones = time.perf_counter()
ones_array = np.ones((500, 500))
ones_seconds = time.perf_counter() - start_ones

print(f"np.empty: {empty_seconds:.8f} seconds")
print(f"np.ones : {ones_seconds:.8f} seconds")

**Benchmark caution:** This is only a quick demonstration. Results vary with hardware, memory allocation state, operating system, NumPy build, and system load. Do not memorize a fixed speed ratio from one run.

## 2. Random arrays with `np.random.rand()`

The class introduced random values as an important building block for machine learning, including the idea of initializing model weights with random values.

`np.random.rand(d0, d1, ...)` creates an array with the requested dimensions, containing random floating-point values in the interval **[0, 1)**:
- 0 can occur;
- 1 is excluded;
- values are drawn from a uniform distribution.

In [ ]:
random_array = np.random.rand(2, 8)
print(random_array)
print("shape:", random_array.shape)
print("minimum:", random_array.min())
print("maximum:", random_array.max())

### Important clarification about the distribution

The recording briefly refers to a *normal distribution* while describing `np.random.rand()`. That is not correct for this function: `np.random.rand()` uses a **uniform distribution** over [0, 1).

A normal (Gaussian) distribution is a different distribution and is not what `np.random.rand()` generates. The class's key practical point remains: this function creates random values between 0 (inclusive) and 1 (exclusive), often used to demonstrate random initialization.

## 3. NumPy array attributes

The class covered several attributes that let you inspect an array without changing its values.

### 3.1 `.shape` — size along each axis

For a 2D array, `.shape` returns `(number_of_rows, number_of_columns)`. For higher-dimensional arrays, it returns one size for each axis.

In [ ]:
arr = np.array([[1, 2, 3],
                [4, 5, 6]])

print(arr)
print("shape:", arr.shape)
print("rows:", arr.shape[0])
print("columns:", arr.shape[1])
print("first element type:", type(arr[0, 0]))

### 3.2 `.ndim` — number of dimensions (axes)

`.ndim` tells you how many axes the array has. A 2-row, 3-column matrix is 2D.

In [ ]:
print(arr.ndim)

### 3.3 `.size` — total number of elements

`.size` returns the total number of elements across all dimensions. It is different from `len(array)`, which returns the length of the first axis.

In [ ]:
print("len(arr):", len(arr))  # number of rows: 2
print("arr.size:", arr.size) # total elements: 6

### 3.4 `.dtype` — data type of the elements

A regular NumPy array has one common dtype for its elements. NumPy may convert mixed input values to a common dtype when the array is created.

In [ ]:
print(arr.dtype)

### 3.5 `.itemsize` — bytes used by one element

`.itemsize` reports the number of bytes used by each element. For example, an `int64` element uses 8 bytes, which is 64 bits.

In [ ]:
print("dtype:", arr.dtype)
print("itemsize:", arr.itemsize, "bytes")
print("bits per element:", arr.itemsize * 8)

### 3.6 `.nbytes` — total bytes used by array elements

`.nbytes` is the total memory occupied by the array's element buffer. For a simple contiguous array, it is `size × itemsize`.

In [ ]:
print("number of elements:", arr.size)
print("bytes per element:", arr.itemsize)
print("total element bytes:", arr.nbytes)
print("size * itemsize:", arr.size * arr.itemsize)

**Clarification:** `.nbytes` reports the bytes used by the array elements, not every bit of Python-object overhead associated with the array object itself.

### 3.7 `.T` — transpose

Transpose swaps the axes of a 2D matrix: rows become columns and columns become rows. A matrix with shape `(2, 3)` becomes shape `(3, 2)`.

In [ ]:
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])

print("Original:")
print(matrix)
print("Original shape:", matrix.shape)

transposed = matrix.T
print("Transposed:")
print(transposed)
print("Transposed shape:", transposed.shape)

Dry run:

- Original row 1 `[1, 2, 3]` becomes the first column.
- Original row 2 `[4, 5, 6]` becomes the second column.
- Shape changes from `(2, 3)` to `(3, 2)`.

For a 1D array, `.T` does not turn it into a column vector; it remains 1D. Use `reshape(-1, 1)` when you explicitly need a 2D column vector.

## 4. Reshaping arrays with `.reshape()`

`reshape()` changes the structure (shape) of an array without changing the sequence of its data values. The total number of elements must remain the same.

For example, 6 values can be arranged as:
- `(2, 3)` — 2 rows and 3 columns;
- `(3, 2)` — 3 rows and 2 columns;
- `(1, 6)` — 1 row and 6 columns;
- `(6, 1)` — 6 rows and 1 column.

But 6 values cannot be reshaped to `(4, 2)`, because that shape requires 8 elements.

In [ ]:
values = np.array([1, 2, 3, 4, 5, 6])
print("Original:", values)
print("Original shape:", values.shape)

print("\n2 rows x 3 columns:")
print(values.reshape(2, 3))

print("\n3 rows x 2 columns:")
print(values.reshape(3, 2))

print("\n1 row x 6 columns:")
print(values.reshape(1, 6))

print("\n6 rows x 1 column:")
print(values.reshape(6, 1))

### 4.1 A 1D shape versus a 2D one-row shape

These two shapes are different:

- `(6,)` means a 1D array with 6 elements.
- `(1, 6)` means a 2D array with 1 row and 6 columns.

A 1D array does not have a row axis and a column axis. Reshaping to `(1, 6)` explicitly adds those two-dimensional axes.

In [ ]:
one_d = np.array([1, 2, 3, 4, 5, 6])
one_row = one_d.reshape(1, 6)

print("one_d shape:", one_d.shape, "| ndim:", one_d.ndim)
print("one_row shape:", one_row.shape, "| ndim:", one_row.ndim)

### 4.2 Let NumPy infer one dimension with `-1`

Use `-1` for one dimension when you know the other dimension but do not want to calculate the remaining size manually. NumPy infers it from the total number of elements.

For 9 elements:
- `reshape(3, -1)` means 3 rows; NumPy infers 3 columns.
- `reshape(-1, 3)` means 3 columns; NumPy infers 3 rows.

Only one dimension can be inferred using `-1` in a reshape call.

In [ ]:
nine_values = np.array([1, 2, 3, 4, 5, 6, 56, 100, 789])

as_three_rows = nine_values.reshape(3, -1)
as_three_columns = nine_values.reshape(-1, 3)

print(as_three_rows)
print("shape:", as_three_rows.shape)

print(as_three_columns)
print("shape:", as_three_columns.shape)

If the requested dimensions cannot contain the same number of elements, NumPy raises a `ValueError`. For example, 9 values cannot be reshaped to `(2, -1)` because 9 is not divisible by 2.

In [ ]:
# This is an intentionally invalid reshape; run only if you want to see the ValueError.
# nine_values.reshape(2, -1)

## 5. Reshape and views — why changing one can change the other

The class demonstrated that reshaping often returns a **view** of the same underlying data rather than a completely independent copy.

A view has a different way of presenting or organizing the data, but it can share the same underlying memory. Therefore, changing an element through the view can also be visible through the original array.

In [ ]:
original = np.array([1, 2, 3, 4, 5, 6])
reshaped_view = original.reshape(1, 6)

print("Before update:")
print("original:", original)
print("view:", reshaped_view)

reshaped_view[0, 0] = 100

print("\nAfter updating the view:")
print("original:", original)
print("view:", reshaped_view)

### Check whether two arrays share memory

Use `np.shares_memory(a, b)` to test whether two arrays share at least some memory. For this simple reshape example, the result is normally `True`.

In [ ]:
print(np.shares_memory(original, reshaped_view))

### Database-view analogy used in class

The instructor compared a NumPy view with a database view:

- A production table may contain columns that a junior employee should not see or modify.
- Rather than maintaining a separate duplicated table and continually synchronizing it, a database view can present a selected projection of the underlying data.
- In NumPy, a reshape view can present the same data in a different shape without copying the underlying elements when a view is possible.

**Important boundary of the analogy:** a NumPy view is an array-memory concept. A database view is a database query object with database-specific rules; it is not simply the same mechanism as a NumPy view. Database views may also have updateability rules that vary by database and query. The analogy is useful for remembering “different presentation, shared underlying data,” but the two features are not identical.

### View is not the same as a deep copy

- **View:** shares underlying data with another array.
- **Copy:** owns separate element data; changing one does not change the other's values.

Use `.copy()` when you need independent data.

In [ ]:
base = np.array([10, 20, 30])
view_of_base = base.reshape(3, 1)
copy_of_base = base.reshape(3, 1).copy()

view_of_base[0, 0] = 999
print("base after view update:", base)

copy_of_base[1, 0] = 777
print("base after copy update:", base)
print("copy:", copy_of_base)

**Clarification:** `reshape()` returns a view when possible, but NumPy may need to make a copy for some layouts or memory orders. Do not assume every reshape always shares memory; check with `np.shares_memory()` if it matters.

## 6. Updating values with indexing and conditions

NumPy supports direct element updates and condition-based updates.

In [ ]:
arr_update = np.array([1, 2, 3, 4])
arr_update[2] = 10
print(arr_update)

### Boolean indexing

An expression such as `arr > 2` produces a Boolean array. Use that mask to select or update all elements where the condition is true.

In [ ]:
arr_update = np.array([1, 2, 3, 4, 5, 6])
mask = arr_update > 2

print("mask:", mask)
print("selected values:", arr_update[mask])

arr_update[arr_update > 2] = 100
print("after update:", arr_update)

The class also showed a modulo condition. `arr % 2 == 0` is true for even numbers, so it can be used to select or update even elements.

In [ ]:
even_example = np.array([1, 2, 3, 4, 5, 6])
even_example[even_example % 2 == 0] = 100
print(even_example)

## 7. `np.where()` — find the positions that satisfy a condition

`np.where(condition)` returns the indices where the condition is true. For a 1D array, the returned result is a tuple containing one index array.

In [ ]:
numbers = np.array([10, 15, 20, 25, 30])

indices = np.where(numbers > 20)
print("where result:", indices)
print("matching values:", numbers[indices])

Dry run:

- Values greater than 20 are 25 and 30.
- Their zero-based positions are 3 and 4.
- `np.where(numbers > 20)` returns `(array([3, 4]),)`.
- `numbers[indices]` retrieves the values at those positions.

The tuple wrapper is important: `indices[0]` is the actual NumPy array of positions for this 1D case.

In [ ]:
index_array = indices[0]
index_list = index_array.tolist()

print("NumPy index array:", index_array)
print("Python list:", index_list)

### 7.1 Student-percentage example from class

The class used a nested dictionary of students. Each dictionary entry has:
- a student name as the key;
- a value containing subject marks and a final percentage.

The goal is to use NumPy to find the positions of students whose final percentage is greater than a threshold, then map those positions back to the dictionary iteration order.

In [ ]:
students = {'Liam': {'marks': {'Math': 65,
                    'Science': 62,
                    'English': 83,
                    'History': 56,
                    'Geography': 55},
          'final_percentage': 64.2},
 'Noah': {'marks': {'Math': 52,
                    'Science': 75,
                    'English': 97,
                    'History': 100,
                    'Geography': 70},
          'final_percentage': 78.8},
 'Oliver': {'marks': {'Math': 89,
                      'Science': 100,
                      'English': 94,
                      'History': 74,
                      'Geography': 54},
            'final_percentage': 82.2},
 'Elijah': {'marks': {'Math': 61,
                      'Science': 94,
                      'English': 91,
                      'History': 88,
                      'Geography': 80},
            'final_percentage': 82.8},
 'William': {'marks': {'Math': 95,
                       'Science': 55,
                       'English': 60,
                       'History': 97,
                       'Geography': 68},
             'final_percentage': 75.0},
 'James': {'marks': {'Math': 58,
                     'Science': 60,
                     'English': 87,
                     'History': 53,
                     'Geography': 94},
           'final_percentage': 70.4},
 'Benjamin': {'marks': {'Math': 92,
                        'Science': 93,
                        'English': 70,
                        'History': 68,
                        'Geography': 97},
              'final_percentage': 84.0},
 'Lucas': {'marks': {'Math': 79,
                     'Science': 67,
                     'English': 52,
                     'History': 58,
                     'Geography': 73},
           'final_percentage': 65.8},
 'Henry': {'marks': {'Math': 94,
                     'Science': 68,
                     'English': 56,
                     'History': 87,
                     'Geography': 53},
           'final_percentage': 71.6},
 'Alexander': {'marks': {'Math': 73,
                         'Science': 81,
                         'English': 73,
                         'History': 68,
                         'Geography': 57},
               'final_percentage': 70.4},
 'Emma': {'marks': {'Math': 76,
                    'Science': 97,
                    'English': 92,
                    'History': 79,
                    'Geography': 74},
          'final_percentage': 83.6},
 'Olivia': {'marks': {'Math': 96,
                      'Science': 59,
                      'English': 55,
                      'History': 90,
                      'Geography': 79},
            'final_percentage': 75.8},
 'Ava': {'marks': {'Math': 86,
                   'Science': 58,
                   'English': 57,
                   'History': 99,
                   'Geography': 60},
         'final_percentage': 72.0},
 'Isabella': {'marks': {'Math': 79,
                        'Science': 50,
                        'English': 52,
                        'History': 79,
                        'Geography': 56},
              'final_percentage': 63.2},
 'Sophia': {'marks': {'Math': 67,
                      'Science': 85,
                      'English': 93,
                      'History': 64,
                      'Geography': 93},
            'final_percentage': 80.4},
 'Charlotte': {'marks': {'Math': 74,
                         'Science': 72,
                         'English': 86,
                         'History': 81,
                         'Geography': 73},
               'final_percentage': 77.2},
 'Mia': {'marks': {'Math': 65,
                   'Science': 55,
                   'English': 88,
                   'History': 50,
                   'Geography': 63},
         'final_percentage': 64.2},
 'Amelia': {'marks': {'Math': 98,
                      'Science': 76,
                      'English': 74,
                      'History': 87,
                      'Geography': 77},
            'final_percentage': 82.4},
 'Harper': {'marks': {'Math': 55,
                      'Science': 99,
                      'English': 61,
                      'History': 81,
                      'Geography': 100},
            'final_percentage': 79.2},
 'Evelyn': {'marks': {'Math': 77,
                      'Science': 54,
                      'English': 58,
                      'History': 72,
                      'Geography': 74},
            'final_percentage': 67.0}}

In [ ]:
student_percentages = np.array([
    info["final_percentage"] for name, info in students.items()
])

elite_indices_tuple = np.where(student_percentages > 80)
elite_indices = elite_indices_tuple[0].tolist()

print("percentages:", student_percentages)
print("where tuple:", elite_indices_tuple)
print("matching indices:", elite_indices)

### Map the indices back to student names

A Python dictionary is accessed by its keys, not by positional indexes. To connect the NumPy positions to dictionary entries, the class used `enumerate(students.items())`.

`enumerate()` supplies a counter (0, 1, 2, ...) alongside each `(name, info)` pair. If the counter is in the list of matching positions, print the corresponding name.

In [ ]:
for i, (name, info) in enumerate(students.items()):
    if i in elite_indices:
        print(name, info["final_percentage"])

### A simpler Python alternative for this particular dictionary

The instructor also showed that this particular task can be solved directly by checking the dictionary values. This is easier to read for a small nested dictionary:

In [ ]:
for name, info in students.items():
    if info["final_percentage"] > 80:
        print(name, info["final_percentage"])

**Why learn `np.where()` then?** The point of the class example was to learn how to locate positions that satisfy a numerical condition. In larger numerical arrays, finding all matching positions without writing a Python loop over every element is useful. The dictionary mapping step is a separate operation because dictionaries use keys.

## 8. Python list addition versus NumPy array addition

A Python list's `+` operator concatenates lists. NumPy array `+` performs element-wise numerical addition when shapes are compatible.

In [ ]:
a_list = [1, 2, 3]
b_list = [4, 5, 6]

print("Python lists:", a_list + b_list)

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("NumPy addition:", a + b)
print("NumPy multiplication:", a * b)
print("NumPy subtraction:", a - b)

Dry run:
- `a + b` → `[1+4, 2+5, 3+6]` → `[5, 7, 9]`
- `a * b` → `[1×4, 2×5, 3×6]` → `[4, 10, 18]`
- `a - b` → `[1−4, 2−5, 3−6]` → `[-3, -3, -3]`

This is **element-wise** arithmetic. It is not matrix multiplication.

## 9. Dot product — from 1D vectors to matrices

### 9.1 Dot product of two 1D arrays

For two vectors of equal length, the dot product multiplies corresponding elements and adds the products.

For `[1, 2, 3]` and `[4, 5, 6]`:

\[
(1 	imes 4) + (2 	imes 5) + (3 	imes 6) = 4 + 10 + 18 = 32
\]

In [ ]:
vector_a = np.array([1, 2, 3])
vector_b = np.array([4, 5, 6])

print("element-wise products:", vector_a * vector_b)
print("dot product:", np.dot(vector_a, vector_b))

### 9.2 Matrix multiplication shape rule

For 2D matrices:
- Matrix A has shape `(M, N)`.
- Matrix B has shape `(N, P)`.
- The inner dimensions `N` must match.
- The result has shape `(M, P)`.

In short: **columns of the first matrix must equal rows of the second matrix.**

Examples:
- `(3, 5) · (5, 3)` is valid and produces `(3, 3)`.
- `(10, 2) · (2, 8)` is valid and produces `(10, 8)`.
- `(2, 3) · (2, 3)` is not valid because the inner dimensions 3 and 2 do not match.

In [ ]:
A = np.array([[1, 2],
              [3, 4]])       # shape (2, 2)

B = np.array([[1, 2],
              [3, 4]])       # shape (2, 2)

product = np.dot(A, B)
print(product)
print("shape:", product.shape)

### 9.3 How the 2 × 2 dot product is calculated

For the matrices above:

\[
A =
egin{bmatrix}
1 & 2 \\
3 & 4
\end{bmatrix}
\qquad
B =
egin{bmatrix}
1 & 2 \\
3 & 4
\end{bmatrix}
\]

Each output cell is the dot product of one row from A and one column from B:

- Row 1 × Column 1: `1×1 + 2×3 = 7`
- Row 1 × Column 2: `1×2 + 2×4 = 10`
- Row 2 × Column 1: `3×1 + 4×3 = 15`
- Row 2 × Column 2: `3×2 + 4×4 = 22`

Therefore the result is `[[7, 10], [15, 22]]`.

### 9.4 Use transpose to make shapes compatible

The class demonstrated two arrays with shape `(2, 3)`. Their dot product is not valid as written. Transposing the second array changes its shape to `(3, 2)`, making the inner dimensions match.

In [ ]:
a_1 = np.array([[1, 2, 3],
                [4, 5, 6]])

b_1 = np.array([[1, 2, 3],
                [4, 5, 6]])

print("A shape:", a_1.shape)
print("B shape:", b_1.shape)
print("B.T shape:", b_1.T.shape)

dot_after_transpose = np.dot(a_1, b_1.T)
print(dot_after_transpose)
print("result shape:", dot_after_transpose.shape)

The original `np.dot(a_1, b_1)` example is intentionally invalid because both arrays have shape `(2, 3)`. It is commented out below so the notebook can run from top to bottom.

In [ ]:
# Expected ValueError: shapes (2,3) and (2,3) not aligned
# np.dot(a_1, b_1)

### Clarification about 1D dot products

For 1D arrays, `np.dot(vector_a, vector_b)` returns a scalar. You do not need to transpose either vector. It is better to understand this as NumPy's defined vector dot-product operation, rather than assuming that NumPy literally transposes an array internally.

## 10. Practical use case: calculate sales across states

The class used a simplified Amazon-style example:
- each row in `shop_sold_amazon` represents a state;
- each column represents a product;
- each value is the number of units sold;
- `price_amazon` stores the per-unit price for each state and product.

The goal is to calculate revenue for each state by multiplying each product's quantity by its corresponding unit price and adding across products.

In [ ]:
shop_sold_amazon = np.array([
    [100, 200, 20, 30, 5],   # state 1: units sold for 5 products
    [200, 400, 40, 60, 10]   # state 2: units sold for 5 products
])

price_amazon = np.array([
    [30, 10, 1000, 10, 200],   # state 1: price per unit
    [34, 12, 1100, 11, 220]    # state 2: price per unit
])

print("quantity shape:", shop_sold_amazon.shape)
print("price shape:", price_amazon.shape)

### 10.1 Multiply quantities by the matching prices

The quantity array has shape `(2, 5)`. The price array also has shape `(2, 5)`. To use matrix multiplication and calculate the sum of each state's five product revenues, transpose the price array to `(5, 2)`.

Then:
- quantity matrix: `(2, 5)`
- transposed price matrix: `(5, 2)`
- dot-product result: `(2, 2)`

The diagonal cells are the intended state-to-same-state revenue totals. The off-diagonal cells are cross-state combinations that this example does not need.

In [ ]:
sales = np.dot(shop_sold_amazon, price_amazon.T)

print(sales)
print("sales shape:", sales.shape)

### 10.2 Extract the diagonal total using an identity matrix

The class used `np.eye(2)` to create a 2 × 2 identity matrix. Element-wise multiplication by the identity matrix keeps the main diagonal values and turns off-diagonal values into zero. Then `np.sum()` adds the remaining values.

In [ ]:
diagonal_mask = np.eye(2)
print(diagonal_mask)

diagonal_only = sales * diagonal_mask
print(diagonal_only)

total_sales = np.sum(diagonal_only)
print("total sales:", total_sales)

### 10.3 A direct alternative: `np.trace()`

`np.trace(array)` returns the sum of the elements along the main diagonal. For this example it gives the same answer as masking with an identity matrix and then summing.

In [ ]:
trace_total = np.trace(sales)
print("trace total:", trace_total)

### 10.4 Another way: element-wise quantity × price, then sum

Because the quantity and price arrays are aligned row-by-row and column-by-column, the total can also be computed directly by element-wise multiplication followed by summation.

In [ ]:
direct_total = np.sum(shop_sold_amazon * price_amazon)
print("direct total:", direct_total)

print("All three totals match:",
      total_sales == trace_total == direct_total)

**Important interpretation:** The diagonal extraction is needed for the particular matrix-multiplication route used in the class. It is not required if the goal is simply to calculate total revenue and the quantity and price arrays already align element by element; in that case, the direct element-wise method is simpler.

## 11. `np.eye()` and `np.fliplr()` — main diagonal and flipped diagonal

`np.eye(n)` creates an identity matrix with ones on the main diagonal. The class then used `np.fliplr()` to flip the matrix from left to right, producing ones on the opposite (anti-)diagonal.

In [ ]:
identity = np.eye(4)
flipped_identity = np.fliplr(identity)

print("Main diagonal:")
print(identity)

print("Flipped left-to-right:")
print(flipped_identity)

For a 2 × 2 matrix, the main diagonal positions are `(0, 0)` and `(1, 1)`. The opposite diagonal positions are `(0, 1)` and `(1, 0)`.

The class mentioned that a flipped diagonal can be useful when you want to select or sum values along the opposite diagonal. `np.trace()` itself sums the main diagonal; to sum the opposite diagonal, first flip the array left-to-right and then take the trace.

In [ ]:
sample_matrix = np.array([[10, 20],
                          [30, 40]])

main_diagonal_sum = np.trace(sample_matrix)
opposite_diagonal_sum = np.trace(np.fliplr(sample_matrix))

print("main diagonal sum:", main_diagonal_sum)       # 10 + 40 = 50
print("opposite diagonal sum:", opposite_diagonal_sum) # 20 + 30 = 50

## 12. Quick reference table

| Attribute / function | Purpose | Example |
|---|---|---|
| `arr.shape` | Size of each axis | `(2, 3)` |
| `arr.ndim` | Number of axes | `2` |
| `arr.size` | Total element count | `6` |
| `arr.dtype` | Element data type | `int64` |
| `arr.itemsize` | Bytes per element | `8` |
| `arr.nbytes` | Total element-buffer bytes | `48` |
| `arr.T` | Transpose axes | `(2, 3)` → `(3, 2)` |
| `arr.reshape(...)` | Change shape, same element count | `reshape(2, 3)` |
| `np.shares_memory(a, b)` | Test memory sharing | `True` / `False` |
| `np.where(condition)` | Find matching indices | tuple of index arrays |
| `np.dot(a, b)` | Vector/matrix dot product | scalar or array |
| `np.trace(a)` | Sum main diagonal | scalar |
| `np.fliplr(a)` | Flip 2D array left-to-right | array |
| `np.random.rand(2, 8)` | Random values in [0, 1) | shape `(2, 8)` |
| `np.empty(shape)` | Allocate without initialization | uninitialized array |

## 13. Practice exercises

Try these before opening the solutions.

### Exercise 1 — Array attributes

Create a 3 × 4 integer array and print its:
- shape;
- number of dimensions;
- total number of elements;
- dtype;
- itemsize;
- nbytes.

In [ ]:
# Your attempt

### Exercise 2 — Reshape

Create an array containing the integers 1 through 12. Reshape it into:
- 3 rows and 4 columns;
- 4 rows and 3 columns;
- 2 rows with NumPy inferring the number of columns.

In [ ]:
# Your attempt

### Exercise 3 — View and copy

Create a 1D array, reshape it to 2 × 3, and update the first element through the reshaped array. Check whether the original changed. Then repeat using `.copy()`.

In [ ]:
# Your attempt

### Exercise 4 — `np.where`

Given `scores = np.array([45, 72, 88, 59, 91, 64])`, find the indices and values of scores greater than or equal to 70.

In [ ]:
# Your attempt

### Exercise 5 — Dot product

Given:
\[
A = egin{bmatrix}1 & 2 \\ 3 & 4\end{bmatrix},
\quad
B = egin{bmatrix}5 & 6 \\ 7 & 8\end{bmatrix}
\]
Calculate `np.dot(A, B)` and state the output shape.

In [ ]:
# Your attempt

### Exercise 6 — Main and opposite diagonal

Create a 3 × 3 array with values 1 through 9. Calculate:
- the sum of its main diagonal;
- the sum of its opposite diagonal.

In [ ]:
# Your attempt

## 14. Exercise solutions

In [ ]:
# Exercise 1
exercise_1 = np.arange(1, 13, dtype=np.int32).reshape(3, 4)
print("shape:", exercise_1.shape)
print("ndim:", exercise_1.ndim)
print("size:", exercise_1.size)
print("dtype:", exercise_1.dtype)
print("itemsize:", exercise_1.itemsize)
print("nbytes:", exercise_1.nbytes)

In [ ]:
# Exercise 2
twelve = np.arange(1, 13)
print(twelve.reshape(3, 4))
print(twelve.reshape(4, 3))
print(twelve.reshape(2, -1))

In [ ]:
# Exercise 3
source = np.array([1, 2, 3, 4, 5, 6])
shared_view = source.reshape(2, 3)
shared_view[0, 0] = 100
print("source after view change:", source)
print("shares memory:", np.shares_memory(source, shared_view))

independent_copy = source.reshape(2, 3).copy()
independent_copy[0, 0] = 999
print("source after copy change:", source)
print("independent copy:", independent_copy)

In [ ]:
# Exercise 4
scores = np.array([45, 72, 88, 59, 91, 64])
score_indices = np.where(scores >= 70)
print("indices:", score_indices[0])
print("values:", scores[score_indices])

In [ ]:
# Exercise 5
exercise_A = np.array([[1, 2], [3, 4]])
exercise_B = np.array([[5, 6], [7, 8]])
exercise_product = np.dot(exercise_A, exercise_B)
print(exercise_product)
print("shape:", exercise_product.shape)

In [ ]:
# Exercise 6
three_by_three = np.arange(1, 10).reshape(3, 3)
print(three_by_three)
print("main diagonal:", np.trace(three_by_three))
print("opposite diagonal:", np.trace(np.fliplr(three_by_three)))

## 15. What the recording discussed beyond NumPy syntax

The recording included class logistics, questions from learners, and guidance on learning:
- New learners were advised to cover the previous Python and NumPy sessions because this was a continuation class.
- For hands-on practice, the instructor recommended creating a small project rather than treating coding-challenge sites only as the practice method.
- One suggested workflow was to ask an AI assistant for a project statement that uses selected topics, attempt the project without looking at the solution, and then ask the AI to review whether the required topics were covered.
- The instructor encouraged students to practise and explore in parallel with the course rather than waiting for the full course to finish.
- The roadmap discussed later modules such as Pandas-based data analysis, visualization, databases, Streamlit, statistics, feature engineering, EDA, machine learning, APIs, MLflow, NLP, and big-data processing. This was a roadmap discussion, not material taught as NumPy APIs in this session.

The Q&A also included unrelated questions about Python OOP access modifiers and composition, plus placement preparation. Those are not expanded here because they were not part of the NumPy lesson.

## 16. Class coverage and clarification notes

### Covered in the source notebook and recording
- `np.empty()` and its uninitialized values.
- A timing comparison of `np.empty()` and `np.ones()`.
- `np.random.rand()` for random values.
- Array attributes: `.shape`, `.ndim`, `.size`, `.dtype`, `.itemsize`, `.nbytes`, `.T`.
- Reshaping and inferred dimensions using `-1`.
- Views and shared memory in reshape operations.
- Index-based and condition-based updates.
- `np.where()` and retrieving student names by matching positions.
- Element-wise arithmetic vs dot product.
- Dot product of vectors and matrices, shape compatibility, and transpose.
- Sales calculation using matrix multiplication, diagonal masking, `np.trace()`, and direct element-wise multiplication.
- `np.fliplr()` for flipping a matrix.

### Explicit technical clarifications
1. `np.random.rand()` draws from a uniform distribution on [0, 1), not a normal distribution.
2. `np.empty()` does not erase old values or return an array of guaranteed zeros; its elements are uninitialized.
3. `reshape()` returns a view when possible, but may return a copy depending on memory layout.
4. NumPy views and database views are analogous only at a high level; their actual semantics differ.
5. `.nbytes` reports the array element buffer size, not all Python object overhead.
6. A 1D vector dot product returns a scalar; it is not necessary to describe this as an automatic transpose.
7. The revenue example can be solved directly with element-wise quantity-price multiplication and `sum`; the diagonal method demonstrates the matrix operations taught in class.

### Not covered in detail
The recording mentions future topics such as deeper statistics and machine-learning weight initialization. Those are not expanded into full lessons here.

## 17. Final revision checklist

- [ ] I can explain why `np.empty()` may display arbitrary values.
- [ ] I know when to use `np.empty()` versus `np.zeros()`.
- [ ] I can create random arrays with `np.random.rand()`.
- [ ] I can distinguish `.shape`, `.ndim`, `.size`, `.itemsize`, and `.nbytes`.
- [ ] I can transpose a 2D array using `.T`.
- [ ] I can reshape arrays while preserving the number of elements.
- [ ] I understand `-1` dimension inference.
- [ ] I can explain how a reshape view can share memory with its original array.
- [ ] I can use Boolean indexing and `np.where()`.
- [ ] I can distinguish element-wise multiplication from dot product.
- [ ] I can check dot-product shape compatibility.
- [ ] I can use `np.trace()` to sum a main diagonal.
- [ ] I can use `np.fliplr()` before `np.trace()` to sum the opposite diagonal.

**End of NumPy Class 2 — 31 October 2026.**

## Appendix A — Original class notebook reference

This appendix preserves the supplied notebook's examples for comparison. Main explanations and corrected runnable examples are above. Code cells that intentionally demonstrate an invalid shape operation are commented to keep sequential execution safe.

In [ ]:
import numpy as np

### Original notebook Markdown cell 2

# np.empty(shape)

In [ ]:
np.empty((2,2), dtype=np.int8)

In [ ]:
np.empty((2,2), dtype=np.int8)

In [ ]:
import time

In [ ]:
start_e = time.perf_counter()
np.empty((5,5))
end_e = time.perf_counter()
total_e = end_e - start_e
print(total_e)

In [ ]:
start_we = time.perf_counter()
np.ones((5,5))
end_we = time.perf_counter()
total_we = end_we - start_we
print(total_we)

### Original notebook Markdown cell 8

# np.random.rand(shape)

random values between 0 and 1

- ML weights random initialization

In [ ]:
np.random.rand(2,8)

### Original notebook Markdown cell 10

# Numpy Array attributes

### Original notebook Markdown cell 11

### 1. ndarray.shape

In [ ]:
arr = np.array([[1, 2, 3], [4, 5, 6]])
print(arr)
print(arr.shape)

In [ ]:
print(type(arr[0][0]))

In [ ]:
print(arr.shape[0])
print(arr.shape[1])

### Original notebook Markdown cell 15

### 2. ndarray.ndim

In [ ]:
print(arr.ndim)

### Original notebook Markdown cell 17

### 3. ndarray.size

Returns the total number of elements in the array.

In [ ]:
print(arr.size)

### Original notebook Markdown cell 19

### 4. ndarray.dtype

Tells the data type of the array elements.

In [ ]:
print(arr.dtype)

### Original notebook Markdown cell 21

### 5. ndarray.itemsize

Returns the size in bytes of one element.

1 byte --> 8 bit

int8 --> 8 bit

int64 --> int8 * 8 --> int64

In [ ]:
print(arr.itemsize)

### Original notebook Markdown cell 23

### 6. ndarray.nbytes
Total number of bytes consumed by the array.

In [ ]:
print(arr.nbytes)

### Original notebook Markdown cell 25

### 7. ndarray.T

Returns the transpose of the array.

In [ ]:
print(arr) # 2 rows, 3 column

In [ ]:
print(arr.T)

### Original notebook Markdown cell 28

# Array Reshaping and resizing in numpy

### Original notebook Markdown cell 29

###  1. reshape()

Change Shape Without Changing Data

##### Notes:
The total number of elements must remain the same.

reshape() returns a new view if possible (no copy).

You can use -1 to let NumPy infer one dimension:

In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6])
print(arr.shape)

In [ ]:
# 2x3
# 3X2
# 4x2 -> No
# 6x1 -> Yes

In [ ]:
reshaped_1 = arr.reshape(2, 3)
print(reshaped_1)

In [ ]:
reshaped_2 = arr.reshape(3, 2)
print(reshaped_2)

In [ ]:
reshaped_2 = arr.reshape(1, 6)
print(reshaped_2)
print(reshaped_2.ndim)

In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6, 56, 100, 789])
print(arr.shape)

In [ ]:
reshaped_3 = arr.reshape(3,-1)
print(reshaped_3.shape)

In [ ]:
reshaped_3 = arr.reshape(-1, 3)
print(reshaped_3.shape)

In [ ]:
# Original class example: 9 values cannot be reshaped to (2, -1), because 9 is not divisible by 2.
# reshaped_3 = arr.reshape(2, -1)
# print(reshaped_3.shape)

### Original notebook Markdown cell 39

# testing view logic

In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6])
print(arr.shape)

In [ ]:
reshaped_2 = arr.reshape(1, 6)

In [ ]:
reshaped_2

In [ ]:
reshaped_2[0][0] = 100

In [ ]:
reshaped_2

In [ ]:
arr

### Original notebook Markdown cell 46

### 5. Modifying with Indexing

In [ ]:
arr = np.array([1, 2, 3, 4])
arr[2] = 10
print(arr)

In [ ]:
arr[arr > 2] = 100
# arr[arr%2==0] = 100
print(arr)

### Original notebook Markdown cell 49

### 6. Indexing with np.where()

In [ ]:
arr = np.array([10, 15, 20, 25, 30])

In [ ]:
idx = np.where(arr > 20)

In [ ]:
print(idx)

In [ ]:
print(arr[idx])

In [ ]:
students = {'Liam': {'marks': {'Math': 65,
                    'Science': 62,
                    'English': 83,
                    'History': 56,
                    'Geography': 55},
          'final_percentage': 64.2},
 'Noah': {'marks': {'Math': 52,
                    'Science': 75,
                    'English': 97,
                    'History': 100,
                    'Geography': 70},
          'final_percentage': 78.8},
 'Oliver': {'marks': {'Math': 89,
                      'Science': 100,
                      'English': 94,
                      'History': 74,
                      'Geography': 54},
            'final_percentage': 82.2},
 'Elijah': {'marks': {'Math': 61,
                      'Science': 94,
                      'English': 91,
                      'History': 88,
                      'Geography': 80},
            'final_percentage': 82.8},
 'William': {'marks': {'Math': 95,
                       'Science': 55,
                       'English': 60,
                       'History': 97,
                       'Geography': 68},
             'final_percentage': 75.0},
 'James': {'marks': {'Math': 58,
                     'Science': 60,
                     'English': 87,
                     'History': 53,
                     'Geography': 94},
           'final_percentage': 70.4},
 'Benjamin': {'marks': {'Math': 92,
                        'Science': 93,
                        'English': 70,
                        'History': 68,
                        'Geography': 97},
              'final_percentage': 84.0},
 'Lucas': {'marks': {'Math': 79,
                     'Science': 67,
                     'English': 52,
                     'History': 58,
                     'Geography': 73},
           'final_percentage': 65.8},
 'Henry': {'marks': {'Math': 94,
                     'Science': 68,
                     'English': 56,
                     'History': 87,
                     'Geography': 53},
           'final_percentage': 71.6},
 'Alexander': {'marks': {'Math': 73,
                         'Science': 81,
                         'English': 73,
                         'History': 68,
                         'Geography': 57},
               'final_percentage': 70.4},
 'Emma': {'marks': {'Math': 76,
                    'Science': 97,
                    'English': 92,
                    'History': 79,
                    'Geography': 74},
          'final_percentage': 83.6},
 'Olivia': {'marks': {'Math': 96,
                      'Science': 59,
                      'English': 55,
                      'History': 90,
                      'Geography': 79},
            'final_percentage': 75.8},
 'Ava': {'marks': {'Math': 86,
                   'Science': 58,
                   'English': 57,
                   'History': 99,
                   'Geography': 60},
         'final_percentage': 72.0},
 'Isabella': {'marks': {'Math': 79,
                        'Science': 50,
                        'English': 52,
                        'History': 79,
                        'Geography': 56},
              'final_percentage': 63.2},
 'Sophia': {'marks': {'Math': 67,
                      'Science': 85,
                      'English': 93,
                      'History': 64,
                      'Geography': 93},
            'final_percentage': 80.4},
 'Charlotte': {'marks': {'Math': 74,
                         'Science': 72,
                         'English': 86,
                         'History': 81,
                         'Geography': 73},
               'final_percentage': 77.2},
 'Mia': {'marks': {'Math': 65,
                   'Science': 55,
                   'English': 88,
                   'History': 50,
                   'Geography': 63},
         'final_percentage': 64.2},
 'Amelia': {'marks': {'Math': 98,
                      'Science': 76,
                      'English': 74,
                      'History': 87,
                      'Geography': 77},
            'final_percentage': 82.4},
 'Harper': {'marks': {'Math': 55,
                      'Science': 99,
                      'English': 61,
                      'History': 81,
                      'Geography': 100},
            'final_percentage': 79.2},
 'Evelyn': {'marks': {'Math': 77,
                      'Science': 54,
                      'English': 58,
                      'History': 72,
                      'Geography': 74},
            'final_percentage': 67.0}}

In [ ]:
student_perct = np.array([value["final_percentage"] for key, value in students.items()])

In [ ]:
student_perct

In [ ]:
index_elite = np.where(student_perct>60)

In [ ]:
index_elite

In [ ]:
index_elite = index_elite[0].tolist()

In [ ]:
index_elite

In [ ]:
for i, (name, info) in enumerate(students.items()):
    if i in index_elite:
        print(name)

In [ ]:
for name, info in students.items():
    if info['final_percentage'] > 80:
        print(name)

### Original notebook Markdown cell 63

------------------

In [ ]:
a_list = [1,2,3]
b_list = [4,5,6]

print(a_list + b_list)

In [ ]:
a = np.array([1,2,3])
b = np.array([4,5,6])
print(a+b)

In [ ]:
a*b

In [ ]:
a-b

### Original notebook Markdown cell 68

# dot product

In [ ]:
a = np.array([1,2,3])
b = np.array([4,5,6])

In [ ]:
print(np.dot(a,b))

In [ ]:
npf_1 = np.full((2,3), 10)
print(npf_1)

In [ ]:
npf_2 = np.full((3,2), 10)
print(npf_2)

In [ ]:
np.dot(npf_1, npf_2)

In [ ]:
a_1 = np.array([[1,2,3],[4,5,6]])
b_1 = np.array([[1,2,3],[4,5,6]])

In [ ]:
print(a_1.shape)
print(b_1.shape)

In [ ]:
# Original class example: this is expected to raise a shape-alignment ValueError.
# np.dot(a_1, b_1)

In [ ]:
np.dot(a_1, b_1.T)

### Original notebook Markdown cell 78

# Use case dot product

In [ ]:
shop_sold_amazon = np.array([
    [100, 200, 20, 30, 5], # state-1-product-sold
    [200, 400, 40, 60, 10] # state-2-product-sold
])

print(shop_sold_amazon.shape)

In [ ]:
price_amazon = np.array([
    [30, 10, 1000, 10, 200], # state-1-price
    [34, 12, 1100, 11, 220] # state-2-price
])
print(price_amazon.shape)

In [ ]:
price_amazon.T # original data

### Original notebook Markdown cell 82

Solution-1

In [ ]:
sales = np.dot(shop_sold_amazon, price_amazon.T)

# r1c1
# r1c2 (No Need)
# r2c1 (No Need)
# r2c2

In [ ]:
sales

In [ ]:
# sales[0][0] + sales[1][1]

In [ ]:
diag = np.eye(2)
print(diag)

In [ ]:
sales*diag

In [ ]:
np.sum(sales*diag)

### Original notebook Markdown cell 89

Solution-2

np.trace()

calculates the sum of the elements along the main diagonal of an array

In [ ]:
result = np.trace(sales)

In [ ]:
print(result)

### Original notebook Markdown cell 92

Solution-3

In [ ]:
np.sum(shop_sold_amazon * price_amazon)

### Original notebook Markdown cell 94

# # flip the matrix

In [ ]:
diag = np.eye(2)
diag

In [ ]:
mirror_diag = np.fliplr(diag)
# flip left to right
mirror_diag